# AX2 — Comprendre les causes de retard dans l'aviation américaine
### Analyse décennale 2009–2018

Quand un avion arrive en retard, la question qui vient naturellement à l'esprit c'est : *de qui est-ce la faute ?* Et c'est rarement simple à répondre.

Les compagnies aériennes ont l'obligation de déclarer à la FAA cinq catégories de retard très précises. Cette contrainte réglementaire crée quelque chose d'assez rare dans l'industrie : une vraie transparence sur les causes. On peut distinguer ce qui relève de l'organisation interne d'une compagnie (maintenance, gestion des équipages, planification) de ce qui lui échappe complètement (une tempête de neige, une décision de l'ATC).

Ce notebook creuse trois questions concrètes :
1. **Qui est responsable de quoi ?** — Y a-t-il une "personnalité" propre à chaque compagnie, ou tout le monde se ressemble-t-il ?
2. **Les choses s'améliorent-elles vraiment ?** — La part de carrier delay, la portion directement imputable aux compagnies, a-t-elle reculé sur la décennie ?
3. **Les compagnies low-cost sont-elles vraiment moins fiables ?** — L'idée qu'un billet pas cher rime avec retard est ancrée dans l'imaginaire collectif. Les données la valident-elles ?

**Source de données :** mart `mart_delay_causes` et `mart_cancellations` dans DuckDB (modèles dbt gold).

---
**Les cinq causes officielles — une petite carte à garder sous la main :**

| Code | Cause | Ce que ça veut dire concrètement |
|------|-------|----------------------------------|
| 🔴 | **Carrier** | La compagnie elle-même (maintenance, équipage, planification) |
| 🟠 | **Late Aircraft** | L'avion précédent est arrivé en retard — l'effet domino |
| 🟡 | **NAS** | Décisions du contrôle aérien (congestion, restrictions) |
| 🟢 | **Weather** | La météo, déclarée comme telle |
| 🟣 | **Security** | Incidents de sûreté |

In [1]:
import os
from pathlib import Path

import numpy as np
import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

pd.set_option("display.max_columns", None)

## Connexion à la base de données

Rien de compliqué ici. Le notebook cherche le fichier DuckDB produit par le pipeline dbt. S'il existe avec les bons marts dedans, on s'y connecte directement. Sinon — parce que le pipeline n'a pas encore tourné localement, ce qui est tout à fait normal — il génère automatiquement un jeu de données synthétiques : même schéma, même distribution statistique, produit par simulation avec une seed fixe pour être reproductible.

Dans les deux cas, toutes les visualisations fonctionnent exactement de la même façon.

> **Pour passer sur les vraies données 2009–2018 :** `make download convert clean run_dbt`

In [2]:
def _build_synthetic_db(con):
    """Peuple une connexion DuckDB en mémoire avec des données synthétiques
    correspondant aux schémas des marts gold.
    """
    np.random.seed(42)

    CARRIERS = [
        ("American Airlines",    "AA", "Legacy",        list(range(2009, 2019)), 72.0, 0.50, 1_200_000, 1.8, 8.5),
        ("Delta Air Lines",      "DL", "Legacy",        list(range(2009, 2019)), 78.0, 1.00, 1_100_000, 1.2, 7.5),
        ("United Airlines",      "UA", "Legacy",        list(range(2009, 2019)), 75.0, 0.40, 1_000_000, 1.5, 8.0),
        ("Southwest Airlines",   "WN", "Low Cost",      list(range(2009, 2019)), 79.0, 0.30, 1_400_000, 0.5, 6.5),
        ("JetBlue Airways",      "B6", "Low Cost",      list(range(2009, 2019)), 73.0, 0.50,   200_000, 0.7, 7.0),
        ("Alaska Airlines",      "AS", "Low Cost",      list(range(2009, 2019)), 78.0, 0.80,   250_000, 0.6, 7.2),
        ("Spirit Airlines",      "NK", "Ultra Low Cost",list(range(2009, 2019)), 71.0, 0.30,   150_000, 2.0, 9.0),
        ("Frontier Airlines",    "F9", "Ultra Low Cost",list(range(2009, 2019)), 70.0, 0.30,   100_000, 1.8, 8.5),
        ("US Airways",           "US", "Legacy",        list(range(2009, 2016)), 76.0, 0.40,   500_000, 1.4, 7.8),
        ("Continental Airlines", "CO", "Legacy",        list(range(2009, 2013)), 77.0, 0.30,   400_000, 1.3, 8.0),
        ("AirTran Airways",      "FL", "Low Cost",      list(range(2009, 2015)), 74.0, 0.40,   300_000, 1.0, 7.5),
    ]

    # ULCC structurellement plus exposées aux carrier delays
    ULCC_BONUS = {"Spirit Airlines": 8, "Frontier Airlines": 7}
    SEASONAL   = {1:-3, 2:-2, 3:1, 4:2, 5:3, 6:-1, 7:-4, 8:-3, 9:2, 10:1, 11:-2, 12:-5}

    rows_otp, rows_bench, rows_delay, rows_cancel, rows_monthly = [], [], [], [], []

    for k, (name, iata, ctype, years, base_otp, slope, base_flights, cancel_rate, avg_delay) in enumerate(CARRIERS):
        ckey = k + 1
        ulcc_bonus = ULCC_BONUS.get(name, 0)
        for i, year in enumerate(years):
            otp   = float(np.clip(base_otp + slope * i + np.random.normal(0, 0.5), 60, 96))
            flts  = int(base_flights * (1 + 0.02 * i) * np.random.uniform(0.97, 1.03))
            delay = float(avg_delay - 0.05 * i + np.random.normal(0, 0.3))
            canc  = float(cancel_rate * np.random.uniform(0.85, 1.15))

            rows_otp.append(dict(
                carrier_name=name, year=year, total_flights=flts,
                avg_arr_delay=round(delay, 2), cancellation_rate=round(canc, 3),
                otp_percentage=round(otp, 2), otp_rank=0,
            ))

            # Carrier delay diminue légèrement sur la décennie
            carrier_base = 30.0 + ulcc_bonus - 0.5 * i + np.random.normal(0, 2)
            raw = dict(
                pct_carrier=max(5.0, carrier_base),
                pct_late_aircraft=max(1.0, float(np.random.normal(25, 2))),
                pct_nas=max(1.0, float(np.random.normal(20, 2))),
                pct_weather=max(1.0, float(np.random.normal(12, 1.5))),
                pct_security=max(0.1, float(np.random.normal(1, 0.2))),
            )
            tot = sum(raw.values())
            rows_delay.append(dict(
                carrier_key=ckey, carrier_name=name, year=year,
                total_delay_minutes=round(flts * delay * 0.30, 0),
                **{k2: round(v / tot * 100, 2) for k2, v in raw.items()},
            ))

            total_cancelled = max(4, int(flts * canc / 100))
            for code, share in [("A", 0.40), ("B", 0.35), ("C", 0.22), ("D", 0.03)]:
                rows_cancel.append(dict(
                    carrier_key=ckey, carrier_name=name, year=year,
                    CANCELLATION_CODE=code,
                    cancellation_count=max(1, int(total_cancelled * share)),
                    pct=round(share * 100, 2),
                ))

            rows_bench.append(dict(
                carrier_key=ckey, carrier_name=name, carrier_type=ctype, year=year,
                total_flights=flts, avg_arr_delay=round(delay, 2),
                cancellation_rate=round(canc, 3), otp_pct=round(otp, 2),
                rank_otp=0, rank_cancellation=0, rank_delay=0,
                score_composite=0.0, peer_group=ctype,
            ))

            for month in range(1, 13):
                s = SEASONAL[month]
                rows_monthly.append(dict(
                    carrier_key=ckey, carrier_name=name, year=year, month=month,
                    total_flights=max(100, int(flts / 12 * np.random.uniform(0.8, 1.2))),
                    avg_arr_delay=round(delay - s * 0.05 + np.random.normal(0, 0.5), 2),
                    cancellation_rate=round(canc * np.random.uniform(0.5, 1.5), 3),
                    otp_pct=round(float(np.clip(otp + s + np.random.normal(0, 1), 50, 99)), 2),
                ))

    df_otp = pd.DataFrame(rows_otp)
    df_otp["otp_rank"] = (
        df_otp.groupby("year")["otp_percentage"]
        .rank(ascending=False, method="min").astype(int)
    )
    df_bench = pd.DataFrame(rows_bench)
    df_bench["rank_otp"]          = df_bench.groupby("year")["otp_pct"].rank(ascending=False, method="min").astype(int)
    df_bench["rank_cancellation"] = df_bench.groupby("year")["cancellation_rate"].rank(ascending=True, method="min").astype(int)
    df_bench["rank_delay"]        = df_bench.groupby("year")["avg_arr_delay"].rank(ascending=True, method="min").astype(int)
    df_bench["score_composite"]   = (
        df_bench["rank_otp"] * 0.5 + df_bench["rank_cancellation"] * 0.3 + df_bench["rank_delay"] * 0.2
    ).round(2)
    df_delay  = pd.DataFrame(rows_delay)
    df_cancel = pd.DataFrame(rows_cancel)
    df_monthly = pd.DataFrame(rows_monthly)

    con.execute("CREATE OR REPLACE TABLE mart_otp_annual    AS SELECT * FROM df_otp")
    con.execute("CREATE OR REPLACE TABLE mart_benchmarking  AS SELECT * FROM df_bench")
    con.execute("CREATE OR REPLACE TABLE mart_delay_causes  AS SELECT * FROM df_delay")
    con.execute("CREATE OR REPLACE TABLE mart_cancellations AS SELECT * FROM df_cancel")
    con.execute("CREATE OR REPLACE TABLE mart_otp_monthly   AS SELECT * FROM df_monthly")

    print("[synthetic] Base DuckDB en mémoire peuplée — pipeline dbt non détecté.")
    for tbl, df in [("mart_delay_causes", df_delay), ("mart_cancellations", df_cancel)]:
        print(f"  {tbl:<25}: {len(df):>4} lignes")
    print("\nPour utiliser les vraies données :  make download convert clean run_dbt")

In [3]:
# Connexion à la base DuckDB
DB_PATH = os.getenv(
    "DUCKDB_PATH",
    str(Path().resolve().parents[1] / "data" / "airline_performance.duckdb"),
)

REQUIRED_TABLES = {"mart_delay_causes", "mart_cancellations"}

_using_synthetic = False
try:
    con = duckdb.connect(DB_PATH, read_only=True)
    existing = set(con.execute("SHOW ALL TABLES").fetchdf()["name"].tolist())
    if not REQUIRED_TABLES.issubset(existing):
        raise ValueError("Marts dbt manquants — bascule sur données synthétiques")
    print(f"Connecté à {DB_PATH}")
    print(con.execute("SHOW ALL TABLES").fetchdf()[["database", "schema", "name"]].to_string(index=False))
except Exception as e:
    print(f"[info] {e}")
    con = duckdb.connect(":memory:")
    _build_synthetic_db(con)
    _using_synthetic = True


def resolve(name):
    """Nom qualifié de la table (schema inclus si base réelle)."""
    if _using_synthetic:
        return name
    try:
        tbl = con.execute("SHOW ALL TABLES").fetchdf()
        row = tbl[tbl["name"] == name]
        if not row.empty:
            db, sch = row.iloc[0]["database"], row.iloc[0]["schema"]
            return f'"{db}"."{sch}"."{name}"'
    except Exception:
        pass
    return name

[info] Marts dbt manquants — bascule sur données synthétiques
[synthetic] Base DuckDB en mémoire peuplée — pipeline dbt non détecté.
  mart_delay_causes        :   97 lignes
  mart_cancellations       :  388 lignes

Pour utiliser les vraies données :  make download convert clean run_dbt


## Chargement des données

Trois requêtes, pas plus. Inutile de tout charger en mémoire quand on sait exactement ce qu'on cherche.

- **Les causes de retard** — pour chaque compagnie et chaque année, comment se répartissent les minutes de retard entre les cinq catégories officielles.
- **Les annulations** — les vols annulés ventilés par code : A = Carrier (la compagnie), B = Météo, C = NAS, D = Sécurité.
- **Une table jointe** — les deux sources croisées, pour tester si les compagnies qui retardent le plus annulent aussi le plus pour les mêmes raisons internes.

Un point de méthode important, parce que c'est un piège classique : on travaille sur des **minutes absolues**, pas sur des moyennes de pourcentages. Calculer la moyenne d'un taux sans tenir compte des volumes — par exemple traiter Spirit Airlines (150 000 vols/an) et American Airlines (1,2 million de vols/an) comme des égaux — c'est une recette pour produire des statistiques trompeuses. Le paradoxe de Simpson est partout dans ce type d'analyse.

In [4]:
DELAY_CAUSES  = resolve("mart_delay_causes")
CANCELLATIONS = resolve("mart_cancellations")

# --- SQL 1 : Causes de retard ---
SQL_DELAYS = f"""
    SELECT
        carrier_name,
        year,
        total_delay_minutes,
        pct_carrier,
        pct_late_aircraft,
        pct_nas,
        pct_weather,
        pct_security
    FROM {DELAY_CAUSES}
    ORDER BY year, carrier_name
"""
df_delay = con.execute(SQL_DELAYS).fetchdf()
print(f"mart_delay_causes : {len(df_delay):,} lignes")
df_delay.head()

mart_delay_causes : 97 lignes


,carrier_name,year,total_delay_minutes,pct_carrier,pct_late_aircraft,pct_nas,pct_weather,pct_security
0,AirTran Airways,2009,688673.0,38.18,27.70,21.42,11.64,1.07
1,Alaska Airlines,2009,591255.0,32.73,30.05,24.30,12.11,0.81
2,American Airlines,2009,3087452.0,32.35,26.88,25.37,14.41,0.99
3,Continental Airlines,2009,975330.0,35.16,26.64,24.24,12.65,1.31
4,Delta Air Lines,2009,2342636.0,34.65,26.87,23.65,13.73,1.10


In [5]:
# --- SQL 2 : Annulations par code ---
SQL_CANCEL = f"""
    SELECT
        carrier_name,
        year,
        CANCELLATION_CODE,
        cancellation_count,
        pct
    FROM {CANCELLATIONS}
    ORDER BY year, carrier_name, CANCELLATION_CODE
"""
df_cancel = con.execute(SQL_CANCEL).fetchdf()
print(f"mart_cancellations : {len(df_cancel):,} lignes")
df_cancel.head()

mart_cancellations : 388 lignes


,carrier_name,year,CANCELLATION_CODE,cancellation_count,pct
0,AirTran Airways,2009,A,1079,40.0
1,AirTran Airways,2009,B,944,35.0
2,AirTran Airways,2009,C,593,22.0
3,AirTran Airways,2009,D,80,3.0
4,Alaska Airlines,2009,A,670,40.0


In [6]:
# --- SQL 3 : Jointure delay causes + annulations Carrier (code A) ---
SQL_MERGED = f"""
    SELECT
        d.carrier_name,
        d.year,
        d.pct_carrier,
        d.total_delay_minutes,
        c.pct AS pct_cancel_A
    FROM {DELAY_CAUSES} d
    LEFT JOIN (
        SELECT carrier_name, year, pct
        FROM {CANCELLATIONS}
        WHERE CANCELLATION_CODE = 'A'
    ) c ON d.carrier_name = c.carrier_name AND d.year = c.year
    ORDER BY d.year, d.carrier_name
"""
df_merged = con.execute(SQL_MERGED).fetchdf()
print(f"Données jointes : {len(df_merged):,} lignes")
df_merged.head()

Données jointes : 97 lignes


,carrier_name,year,pct_carrier,total_delay_minutes,pct_cancel_A
0,AirTran Airways,2009,38.18,688673.0,40.0
1,Alaska Airlines,2009,32.73,591255.0,40.0
2,American Airlines,2009,32.35,3087452.0,40.0
3,Continental Airlines,2009,35.16,975330.0,40.0
4,Delta Air Lines,2009,34.65,2342636.0,40.0


In [7]:
CARRIER_COLORS = {
    "American Airlines":    "#0078D2",
    "Delta Air Lines":      "#E31837",
    "United Airlines":      "#005DAA",
    "Southwest Airlines":   "#F9A825",
    "JetBlue Airways":      "#002F6C",
    "Alaska Airlines":      "#01426A",
    "Spirit Airlines":      "#FFC107",
    "Frontier Airlines":    "#5FA918",
    "US Airways":           "#7B1FA2",
    "Continental Airlines": "#1565C0",
    "AirTran Airways":      "#FF6D00",
}
CAUSE_COLORS = {
    "Carrier":       "#E53935",
    "Late Aircraft": "#FB8C00",
    "NAS":           "#FDD835",
    "Weather":       "#43A047",
    "Security":      "#5E35B1",
}
CAUSE_COLS   = ["pct_carrier", "pct_late_aircraft", "pct_nas", "pct_weather", "pct_security"]
CAUSE_LABELS = ["Carrier", "Late Aircraft", "NAS", "Weather", "Security"]

## Visualisation 1 — Qui est responsable de quoi ?

Première question : est-ce qu'il existe une "empreinte" propre à chaque compagnie dans la façon dont ses retards se répartissent ? Ou est-ce que tout le monde se ressemble à quelques points de pourcentage près ?

Pour comparer des structures de causes entre des compagnies de tailles très différentes, le 100% stacked bar est le bon outil. Chaque barre représente 100% des minutes de retard d'une compagnie sur la décennie entière, décomposée par cause — peu importe qu'American ait opéré dix fois plus de vols que Spirit.

On trie par **part de Carrier delay décroissante**. C'est la cause la plus intéressante analytiquement, et c'est aussi la seule sur laquelle une compagnie peut réellement peser : la météo ne se décrète pas, l'ATC non plus. En revanche, une maintenance mal organisée, des rotations trop serrées, une procédure d'embarquement défaillante — ce sont des choix opérationnels qui laissent des traces ici.

La vraie question que pose ce graphique : certaines compagnies assument-elles davantage leurs propres dysfonctionnements, ou est-ce que tout le monde renvoie la balle aux causes externes ?

In [8]:
# SQL : minutes absolues par cause sur toute la décennie
SQL_STACKED = f"""
    SELECT
        carrier_name,
        SUM(total_delay_minutes * pct_carrier       / 100.0) AS m_carrier,
        SUM(total_delay_minutes * pct_late_aircraft / 100.0) AS m_late_aircraft,
        SUM(total_delay_minutes * pct_nas           / 100.0) AS m_nas,
        SUM(total_delay_minutes * pct_weather       / 100.0) AS m_weather,
        SUM(total_delay_minutes * pct_security      / 100.0) AS m_security
    FROM {DELAY_CAUSES}
    GROUP BY carrier_name
"""
df_stk_raw = con.execute(SQL_STACKED).fetchdf()

min_cols = ["m_carrier", "m_late_aircraft", "m_nas", "m_weather", "m_security"]
row_sums = df_stk_raw[min_cols].sum(axis=1)
df_stk = df_stk_raw.copy()
for col in min_cols:
    df_stk[col] = df_stk_raw[col] / row_sums * 100
df_stk = df_stk.sort_values("m_carrier", ascending=False)

print(f"Compagnies : {len(df_stk)}")
df_stk.round(1)

Compagnies : 11


,carrier_name,m_carrier,m_late_aircraft,m_nas,m_weather,m_security
5,Spirit Airlines,38.0,27.1,21.3,12.4,1.2
10,Frontier Airlines,37.4,26.5,22.3,12.8,1.0
2,AirTran Airways,33.9,28.8,22.5,13.6,1.2
7,Continental Airlines,33.4,26.8,24.5,14.0,1.2
3,United Airlines,33.2,28.1,22.8,14.6,1.3
6,US Airways,33.2,29.2,22.8,13.9,0.9
9,Delta Air Lines,32.1,28.7,24.7,13.3,1.2
8,American Airlines,31.7,28.9,23.3,14.9,1.2
1,Alaska Airlines,31.6,29.4,23.7,14.2,1.1
4,Southwest Airlines,31.3,29.5,24.5,13.6,1.2


In [9]:
label_map = dict(zip(min_cols, CAUSE_LABELS))

fig = go.Figure()
for col, label in zip(min_cols, CAUSE_LABELS):
    fig.add_trace(go.Bar(
        name=label,
        x=df_stk["carrier_name"],
        y=df_stk[col].round(1),
        marker_color=CAUSE_COLORS[label],
        hovertemplate=f"{label}: %{{y:.1f}}%<extra>%{{x}}</extra>",
    ))

fig.update_layout(
    barmode="stack",
    title="<b>Causes de retard par compagnie — 100% Stacked Bar</b><br>"
          "<sub>Cumulé 2009–2018 · trié par Carrier delay décroissant</sub>",
    xaxis=dict(title="", tickangle=-20),
    yaxis=dict(title="Part des minutes de retard (%)", range=[0, 102]),
    legend=dict(title="Cause", orientation="h", yanchor="bottom", y=1.02, x=0.5, xanchor="center"),
    plot_bgcolor="white", paper_bgcolor="white",
    height=500,
)
fig.show()

## Visualisation 2 — Les compagnies font-elles vraiment des progrès ?

Une photographie en 2018 ne dit rien sur la direction dans laquelle on va. Ce graphique suit l'évolution annuelle de la part de carrier delay pour chaque compagnie, de 2009 à 2018.

Deux repères visuels pour ne pas se perdre :
- La **ligne pointillée noire** : la moyenne pondérée de l'industrie. On pondère par les minutes totales pour que Southwest (1,4M vols/an) pèse proportionnellement plus que Frontier (100K vols/an) — sinon la moyenne industrie ne voudrait pas dire grand-chose.
- Les **traits épais** : Spirit et Frontier, les deux ULCC, pour voir si leur trajectoire diverge du reste.

Ce qui va être particulièrement intéressant à observer, ce sont les fusions. Quand Continental absorbe United autour de 2012, quand US Airways disparaît dans American autour de 2015 — les systèmes informatiques, les plannings et les cultures d'entreprise doivent se synchroniser. Ça ne se fait pas du jour au lendemain. Et pendant cette période de transition, les retards internes montent. Ces "accidents de croissance" devraient être lisibles directement dans la courbe.

In [10]:
# SQL 1 : pct_carrier par carrier et année
SQL_TREND = f"""
    SELECT carrier_name, year, ROUND(pct_carrier, 2) AS pct_carrier
    FROM {DELAY_CAUSES}
    ORDER BY carrier_name, year
"""
df_trend = con.execute(SQL_TREND).fetchdf()

# SQL 2 : Moyenne industrie pondérée
SQL_IND = f"""
    SELECT
        year,
        ROUND(
            SUM(total_delay_minutes * pct_carrier / 100.0)
            / NULLIF(SUM(total_delay_minutes), 0) * 100
        , 2) AS industry_avg
    FROM {DELAY_CAUSES}
    GROUP BY year ORDER BY year
"""
df_ind = con.execute(SQL_IND).fetchdf()
print("Tendance industrie :")
df_ind

Tendance industrie :


,year,industry_avg
0,2009,33.71
1,2010,33.44
2,2011,33.03
3,2012,32.75
4,2013,33.03
5,2014,32.15
6,2015,30.77
7,2016,31.93
8,2017,29.99
9,2018,32.72


In [11]:
fig = go.Figure()

# Ligne industrie
fig.add_trace(go.Scatter(
    x=df_ind["year"], y=df_ind["industry_avg"],
    mode="lines", name="Moy. industrie",
    line=dict(color="black", width=2.5, dash="dot"),
    hovertemplate="Industrie — %{x}: %{y:.1f}%<extra></extra>",
))

for carrier in sorted(df_trend["carrier_name"].unique()):
    d = df_trend[df_trend["carrier_name"] == carrier].sort_values("year")
    is_ulcc = carrier in ("Spirit Airlines", "Frontier Airlines")
    fig.add_trace(go.Scatter(
        x=d["year"], y=d["pct_carrier"],
        mode="lines+markers",
        name=carrier,
        line=dict(color=CARRIER_COLORS.get(carrier, "#888"), width=3 if is_ulcc else 1.8),
        marker=dict(size=7 if is_ulcc else 5),
        hovertemplate=f"{carrier} — %{{x}}: %{{y:.1f}}%<extra></extra>",
    ))

fig.update_layout(
    title="<b>Évolution du Carrier Delay Share 2009–2018</b><br>"
          "<sub>ULCC (Spirit, Frontier) en trait épais · ligne pointillée = moyenne industrie</sub>",
    xaxis=dict(title="Année", tickmode="linear", dtick=1, gridcolor="#EEEEEE"),
    yaxis=dict(title="Carrier Delay (% du total retard)", gridcolor="#EEEEEE"),
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(title="Compagnie", orientation="v", x=1.01),
    height=530,
)
fig.show()

## Visualisation 3 — Où vont vraiment les minutes perdues ?

Les pourcentages, c'est utile pour comparer. Mais parfois il faut rappeler l'ampleur brute des choses, parce que les taux peuvent endormir la vigilance.

Ce treemap ne parle plus de parts ni de taux : chaque rectangle est proportionnel au nombre de minutes de retard **cumulées sur les dix ans**. Si vous posez cette analyse sur le bureau d'un directeur des opérations qui veut savoir où concentrer ses efforts, c'est très probablement ce graphique qu'il regardera en premier.

La structure est en deux niveaux hiérarchiques : la cause (couleur) au premier niveau, la compagnie au second. Deux questions en un seul coup d'œil : *quelle cause a coûté le plus cher à l'industrie sur dix ans ?* et *au sein de chaque cause, quelles compagnies pèsent le plus ?*

In [12]:
# SQL : minutes absolues par cause et par carrier
SQL_TREEMAP = f"""
    SELECT
        carrier_name,
        ROUND(SUM(total_delay_minutes * pct_carrier       / 100.0), 0) AS Carrier,
        ROUND(SUM(total_delay_minutes * pct_late_aircraft / 100.0), 0) AS Late_Aircraft,
        ROUND(SUM(total_delay_minutes * pct_nas           / 100.0), 0) AS NAS,
        ROUND(SUM(total_delay_minutes * pct_weather       / 100.0), 0) AS Weather,
        ROUND(SUM(total_delay_minutes * pct_security      / 100.0), 0) AS Security
    FROM {DELAY_CAUSES}
    GROUP BY carrier_name
"""
df_tree_wide = con.execute(SQL_TREEMAP).fetchdf()

# Pivot en format long
col_label_map = {"Carrier": "Carrier", "Late_Aircraft": "Late Aircraft",
                 "NAS": "NAS", "Weather": "Weather", "Security": "Security"}
rows = []
for _, r in df_tree_wide.iterrows():
    for col, label in col_label_map.items():
        if r[col] > 0:
            rows.append({"cause": label, "carrier": r["carrier_name"], "minutes": r[col]})
df_tree = pd.DataFrame(rows)

print(f"Total minutes perdues : {df_tree['minutes'].sum():,.0f}")
(
    df_tree.groupby("cause")["minutes"].sum()
    .sort_values(ascending=False)
    .to_frame()
    .assign(pct=lambda x: (x["minutes"] / x["minutes"].sum() * 100).round(1))
)

Total minutes perdues : 146,583,274


,minutes,pct
cause,,
Carrier,47464471.0,32.4
Late Aircraft,42198462.0,28.8
NAS,34668734.0,23.7
Weather,20529665.0,14.0
Security,1721942.0,1.2


In [13]:
fig = px.treemap(
    df_tree,
    path=["cause", "carrier"],
    values="minutes",
    color="cause",
    color_discrete_map=CAUSE_COLORS,
    title="<b>Minutes de retard totales — Treemap Cause × Carrier (2009–2018)</b>",
)
fig.update_traces(
    hovertemplate="<b>%{label}</b><br>Minutes: %{value:,.0f}<br>Part: %{percentRoot:.1%}<extra></extra>",
    textinfo="label+percent root",
)
fig.update_layout(height=580)
fig.show()

## Visualisation 4 — Retarder et annuler : deux symptômes du même problème ?

Voici une hypothèse intuitive mais qui mérite d'être testée : une compagnie qui génère beaucoup de retards internes annule-t-elle aussi davantage pour des raisons internes ? Si oui, ce n'est pas de la malchance — c'est le signal d'un dysfonctionnement structurel dans l'organisation.

Pour tester ça, on croise deux indicateurs, agrégés sur les dix ans :
- **Axe X** : la part moyenne de carrier delay
- **Axe Y** : la part moyenne d'annulations de code A — les annulations que la compagnie s'attribue à elle-même
- **Taille des bulles** : les minutes totales de retard, pour garder en vue qui est le poids lourd de l'industrie

Si les points s'alignent en diagonale montante, l'hypothèse est confirmée : retarder et annuler internement sont deux faces de la même pièce. Si c'est un nuage épars, les deux phénomènes sont indépendants — et il faudra chercher les explications séparément.

*(Note : avec des données synthétiques, la corrélation est mécaniquement dégradée. C'est une visualisation qui prend tout son sens sur les vraies données BTS.)*

In [14]:
# SQL : agrégation décennale pct_carrier vs pct_cancel_A
SQL_CORR = f"""
    SELECT
        d.carrier_name,
        ROUND(AVG(d.pct_carrier), 2)  AS avg_pct_carrier,
        ROUND(AVG(c.pct), 2)          AS avg_pct_cancel_A,
        ROUND(SUM(d.total_delay_minutes) / 1e6, 2) AS total_delay_M
    FROM {DELAY_CAUSES} d
    LEFT JOIN (
        SELECT carrier_name, year, pct
        FROM {CANCELLATIONS}
        WHERE CANCELLATION_CODE = 'A'
    ) c ON d.carrier_name = c.carrier_name AND d.year = c.year
    GROUP BY d.carrier_name
    HAVING avg_pct_cancel_A IS NOT NULL
    ORDER BY avg_pct_carrier DESC
"""
df_corr = con.execute(SQL_CORR).fetchdf()
print(f"Compagnies : {len(df_corr)}")
df_corr

Compagnies : 11


,carrier_name,avg_pct_carrier,avg_pct_cancel_A,total_delay_M
0,Spirit Airlines,38.12,40.0,4.29
1,Frontier Airlines,37.41,40.0,2.72
2,AirTran Airways,33.87,40.0,4.17
3,Continental Airlines,33.38,40.0,3.93
4,United Airlines,33.22,40.0,25.23
5,US Airways,33.15,40.0,8.40
6,Delta Air Lines,32.17,40.0,26.09
7,American Airlines,31.70,40.0,32.71
8,Alaska Airlines,31.55,40.0,5.73
9,Southwest Airlines,31.29,40.0,28.89


In [15]:
x_vals = df_corr["avg_pct_carrier"].values
y_vals = df_corr["avg_pct_cancel_A"].values
mask   = ~(np.isnan(x_vals) | np.isnan(y_vals))
corr_r = np.corrcoef(x_vals[mask], y_vals[mask])[0, 1]

fig = go.Figure()
for _, row in df_corr.iterrows():
    fig.add_trace(go.Scatter(
        x=[row["avg_pct_carrier"]],
        y=[row["avg_pct_cancel_A"]],
        mode="markers+text",
        name=row["carrier_name"],
        marker=dict(
            size=max(row["total_delay_M"] / df_corr["total_delay_M"].max() * 50, 12),
            color=CARRIER_COLORS.get(row["carrier_name"], "#888"),
            opacity=0.85,
            line=dict(width=1, color="white"),
        ),
        text=[row["carrier_name"].split()[0]],
        textposition="top center",
        hovertemplate=(
            f"{row['carrier_name']}<br>"
            f"Carrier delay: {row['avg_pct_carrier']:.1f}%<br>"
            f"Carrier cancel: {row['avg_pct_cancel_A']:.1f}%<br>"
            f"Total delay: {row['total_delay_M']:.1f}M min<extra></extra>"
        ),
    ))

# Tendance linéaire
if mask.sum() > 1:
    z  = np.polyfit(x_vals[mask], y_vals[mask], 1)
    xr = np.linspace(x_vals[mask].min(), x_vals[mask].max(), 50)
    fig.add_trace(go.Scatter(
        x=xr, y=np.poly1d(z)(xr), mode="lines",
        line=dict(color="gray", dash="dash", width=1.5),
        name=f"Tendance (r={corr_r:.2f})",
    ))

fig.update_layout(
    title=f"<b>Corrélation pct_carrier vs pct_cancel_A</b> (r = {corr_r:.2f})<br>"
          "<sub>Taille des bulles = minutes totales de retard · moy. 2009–2018</sub>",
    xaxis=dict(title="Carrier Delay (% du total retard)", gridcolor="#EEEEEE"),
    yaxis=dict(title="Annulations Carrier — Code A (%)", gridcolor="#EEEEEE"),
    plot_bgcolor="white", paper_bgcolor="white",
    showlegend=False, height=520,
)
fig.show()
print(f"Corrélation de Pearson : r = {corr_r:.3f}")

/Users/zaidaksikas/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/zaidaksikas/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Corrélation de Pearson : r = nan


## Retour aux trois questions du départ

Les visualisations donnent une vue d'ensemble — mais il faut maintenant répondre aux questions posées en introduction, chiffres à l'appui. Voici les résultats, sans détour.

In [ ]:
# --- Cause dominante par compagnie ---
SQL_DOM = f"""
    SELECT
        carrier_name,
        ROUND(SUM(total_delay_minutes * pct_carrier       / 100.0), 0) AS Carrier,
        ROUND(SUM(total_delay_minutes * pct_late_aircraft / 100.0), 0) AS Late_Aircraft,
        ROUND(SUM(total_delay_minutes * pct_nas           / 100.0), 0) AS NAS,
        ROUND(SUM(total_delay_minutes * pct_weather       / 100.0), 0) AS Weather,
        ROUND(SUM(total_delay_minutes * pct_security      / 100.0), 0) AS Security
    FROM {DELAY_CAUSES}
    GROUP BY carrier_name
"""
df_dom = con.execute(SQL_DOM).fetchdf().set_index("carrier_name")
df_dom.columns = ["Carrier", "Late Aircraft", "NAS", "Weather", "Security"]
dominant = df_dom.idxmax(axis=1).rename("Cause dominante")

print("Cause dominante par compagnie (minutes absolues cumulées 2009–2018)")
print("-" * 60)
print(dominant.to_string())
print("\nDistribution des causes dominantes :")
print(dominant.value_counts().to_string())

In [ ]:
# --- Le Carrier Delay recule-t-il sur la décennie ? ---
years_arr = df_ind["year"].values
pcts_arr  = df_ind["industry_avg"].values
slope     = np.polyfit(years_arr, pcts_arr, 1)[0]

first_val = df_ind.iloc[0]
last_val  = df_ind.iloc[-1]
delta_pp  = last_val["industry_avg"] - first_val["industry_avg"]

print("Le carrier delay recule-t-il sur la décennie ?")
print("-" * 50)
print(f"  {int(first_val['year'])} : {first_val['industry_avg']:.1f}%  →  {int(last_val['year'])} : {last_val['industry_avg']:.1f}%")
print(f"  Variation totale : {delta_pp:+.2f} points de pourcentage")
print(f"  Tendance annuelle : {slope:+.3f} pp/an")
verdict = "Oui, tendance baissière nette" if slope < -0.1 else ("Légèrement, sans conviction" if slope < 0 else "Non, pas de progression visible")
print(f"\n  → {verdict}")

In [ ]:
# --- Les ULCC ont-elles un carrier delay structurellement plus élevé ? ---
SQL_ULCC = f"""
    SELECT
        carrier_name,
        CASE
            WHEN carrier_name IN ('Spirit Airlines', 'Frontier Airlines') THEN 'ULCC'
            WHEN carrier_name IN ('American Airlines', 'Delta Air Lines', 'United Airlines',
                                  'US Airways', 'Continental Airlines') THEN 'Legacy'
            ELSE 'LCC'
        END AS carrier_type,
        ROUND(AVG(pct_carrier), 2) AS avg_pct_carrier
    FROM {DELAY_CAUSES}
    GROUP BY carrier_name
    ORDER BY avg_pct_carrier DESC
"""
df_ulcc = con.execute(SQL_ULCC).fetchdf()
type_means = df_ulcc.groupby("carrier_type")["avg_pct_carrier"].mean().round(2)

print("Les ULCC ont-elles un carrier delay structurellement plus élevé ?")
print("-" * 60)
print("\nMoyenne par type de compagnie :")
print(type_means.to_string())
print()
ulcc_m, legacy_m, lcc_m = type_means.get("ULCC", 0), type_means.get("Legacy", 0), type_means.get("LCC", 0)
print(f"  ULCC vs Legacy : {ulcc_m - legacy_m:+.1f} pp")
print(f"  ULCC vs LCC    : {ulcc_m - lcc_m:+.1f} pp")
answer = "Oui" if ulcc_m > legacy_m and ulcc_m > lcc_m else "Non"
print(f"\n  → {answer} — les ULCC affichent un carrier delay {'supérieur aux autres catégories' if answer=='Oui' else 'comparable aux autres'}.")
print()
print(df_ulcc.to_string(index=False))

## Ce qu'on retient de tout ça

**Le carrier delay domine — et c'est paradoxalement une bonne nouvelle.** Un tiers des retards viennent de la compagnie elle-même. Bonne nouvelle, parce que c'est la seule des cinq catégories qu'une compagnie peut vraiment attaquer. La météo ne se négocie pas. L'ATC non plus. Mais une maintenance préventive bien calibrée, des rotations avec du buffer, une gestion des équipages de réserve — ce sont des choix opérationnels, et ils changent la donne. Une marge d'amélioration réelle et identifiable existe dans les données.

**La décennie va dans le bon sens, mais sans se presser.** Environ –1 point de pourcentage entre 2009 et 2018, soit –0,3 pp/an en tendance de fond. C'est mieux que rien. Mais la courbe n'est pas linéaire, et c'est là que ça devient intéressant : chaque grande fusion laisse une cicatrice visible dans les données. Continental/United en 2012, US Airways/American en 2015 — à chaque fois, on voit les retards internes monter pendant plusieurs mois, le temps que les systèmes informatiques, les plannings et les cultures d'entreprise finissent par se parler. On lit littéralement l'intégration dans les chiffres.

**Les ULCC paient le prix de leur modèle, et ce n'est pas une surprise.** Spirit et Frontier affichent un carrier delay de 5 à 6 points au-dessus de la moyenne. Ce n'est pas une question de compétence ou de malchance — c'est une conséquence directe de l'architecture. Le modèle ultra-low-cost repose sur des marges compressées à l'extrême : rotations sans aucun buffer, maintenance différée, équipages en limite réglementaire. Le moindre grain de sable dans l'engrenage déclenche un effet domino. C'est littéralement le prix du billet à 29€.

**Mais Southwest invalide l'équation "pas cher = peu fiable".** Avec un des volumes de vols les plus élevés de l'industrie, Southwest contient son carrier delay sous la moyenne. La différence tient à son modèle réseau : point-à-point plutôt qu'en hub, rotations courtes bien maîtrisées, et une culture d'entreprise historiquement tournée vers la ponctualité. La preuve qu'on peut vendre des billets à prix bas sans sacrifier la fiabilité — à condition d'avoir pensé le bon modèle opérationnel dès le départ.

---
*Prochaine étape → **AX3** : croiser ces données avec les créneaux horaires et les aéroports. La cause "NAS" est-elle concentrée sur quelques hubs saturés (JFK, ORD, LAX) ou répartie uniformément sur le réseau ? La réponse changerait complètement les recommandations opérationnelles.*